# Processing Data and Converting Excel to Parquet

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly
import scipy as sp
import statsmodels as sm
import sklearn as sk
import os
import re
import time
from tqdm import tqdm

Potentially change working directory

In [ ]:
print(os.getcwd())

List files

In [ ]:
print(os.listdir())

Read static reference data

In [ ]:
static_reference_data = {}
for batch_index in tqdm([1,2], desc='Outer Loop'):
    static_reference_data_filenames = os.listdir(f"palate_data_excel/batch_{batch_index}")
    for filename in tqdm(static_reference_data_filenames, desc='Inner Loop', leave=False):
        if ".xlsx" in filename:
            df = pd.read_excel(f"palate_data_excel/batch_{batch_index}/{filename}")
            base_name = re.sub(r'\.xlsx$', '', filename)
            static_reference_data[f"{base_name}_{batch_index}"] = df

Verify there is data for each location id

In [ ]:
locations = static_reference_data['locations_1']['location_id'].value_counts().index.tolist()
locations.extend(static_reference_data['locations_2']['location_id'].value_counts().index.tolist())
locations.sort()
location_filenames = os.listdir("palate_data_excel/batch_1/orders_item_level") + os.listdir("palate_data_excel/batch_2/orders_item_level")
location_data = pd.Series(location_filenames).str.replace('.xlsx', '')
pd.merge(pd.DataFrame(locations, columns=['location_id']), pd.DataFrame(location_data, columns=['location_data']), left_on='location_id', right_on='location_data', how='outer')

Write static reference data to parquet files

In [ ]:
# for base_name, df in tqdm(static_reference_data.items()):
#     for column in df.columns:
#         df[column] = df[column].astype(str)
#     df.to_parquet(base_name + '.parquet')

Read sales data for each restaurant

In [ ]:
restaurant_data = {}
part1_files = {}
# Read restaurant dataframes
for batch_index in [1,2]:
    location_filenames = os.listdir(f"palate_data_excel/batch_{batch_index}/orders_item_level")
    for location_filename in tqdm(location_filenames):
        location_id = re.sub(r'(_part[12])?\.xlsx$', '', location_filename)
        if "part1" in location_filename:
            part1_files[location_id] = location_filename
        else:
            df = pd.read_excel(f"palate_data_excel/batch_{batch_index}/orders_item_level/" + location_filename)
            restaurant_data[location_id] = df
    
# Read dataframes with a part1 and part2 then merge
for location_id, part1_filename in part1_files.items():
    df1 = pd.read_excel(f"palate_data_excel/batch_1/orders_item_level/" + part1_filename)
    part2_filename = f"{location_id}_part2.xlsx"
    df2 = pd.read_excel(f"palate_data_excel/batch_1/orders_item_level/" + part2_filename)
    restaurant_data[location_id] = pd.concat([df1, df2])

Keep largest file split

In [ ]:
largedf_part1 = pd.read_excel(f"palate_data_excel/batch_1/orders_item_level/2HRX9P6HKXA8V_part1.xlsx")
largedf_part2 = pd.read_excel(f"palate_data_excel/batch_1/orders_item_level/2HRX9P6HKXA8V_part2.xlsx")
for column in largedf_part1.columns:
    largedf_part1[column] = largedf_part1[column].astype(str)
    largedf_part2[column] = largedf_part2[column].astype(str)
largedf_part1.to_parquet("2HRX9P6HKXA8V_part1.parquet")
largedf_part2.to_parquet("2HRX9P6HKXA8V_part2.parquet")

Write sales data to parquet files

In [ ]:
# for location_id, df in tqdm(restaurant_data.items()):
#     for column in df.columns:
#         df[column] = df[column].astype(str)
#     df.to_parquet(location_id + ".parquet")